# Linux Package Management

---

## The Core Concept (Lock This In First)

A **package manager** is a tool that automates:
- Downloading software from trusted repositories
- Resolving dependencies (installing everything a package needs)
- Installing, updating, and removing software consistently
- Verifying package integrity via checksums/signatures

Without a package manager, you'd manually download tarballs, compile from source, and hunt down every dependency yourself.

---

## The Two Ecosystems

| Property | Debian/Ubuntu | RedHat/CentOS/Fedora |
|---|---|---|
| Package format | `.deb` | `.rpm` |
| Low-level tool | `dpkg` | `rpm` |
| High-level tool | `apt` / `apt-get` | `yum` / `dnf` |
| Repo config location | `/etc/apt/sources.list` | `/etc/yum.repos.d/` |
| Cache location | `/var/cache/apt/archives/` | `/var/cache/yum/` |
| Lock file | `/var/lib/dpkg/lock` | `/var/run/yum.pid` |

> `apt`/`yum` are the high-level tools you use daily — they handle dependency resolution automatically. `dpkg`/`rpm` are low-level — they install a `.deb`/`.rpm` file directly but won't resolve dependencies.

---

## APT — Ubuntu / Debian

### Essential Commands

```bash
# ─── Update & Upgrade ──────────────────────────────────────────
apt update                          # refresh package index (always run this first)
apt upgrade                         # upgrade all installed packages
apt full-upgrade                    # upgrade + remove obsolete packages

# ─── Install & Remove ─────────────────────────────────────────
apt install nginx                   # install a package
apt install nginx=1.18.0            # install specific version
apt remove nginx                    # remove package (keeps config files)
apt purge nginx                     # remove package + all config files
apt autoremove                      # remove unused dependency packages

# ─── Search & Info ────────────────────────────────────────────
apt search nginx                    # search available packages
apt show nginx                      # show package details (version, deps, size)
apt list --installed                # list all installed packages
apt list --installed | grep nginx   # check if specific package is installed

# ─── Cache Management ─────────────────────────────────────────
apt clean                           # delete downloaded package files
apt autoclean                       # delete only outdated downloaded files
```

### `apt` vs `apt-get` — When to Use Which

| Tool | Use When | Notes |
|---|---|---|
| `apt` | Interactive terminal use | Human-friendly output, progress bars |
| `apt-get` | Scripts and automation | Stable output format, recommended for CI/CD |
| `apt-cache` | Querying package info | `apt-cache search`, `apt-cache show` |

> In Dockerfiles and shell scripts, always use `apt-get` — its output format is guaranteed stable across versions. `apt`'s output can change and break script parsing.

### APT in Dockerfiles (Real MLOps Pattern)

```dockerfile
# Best practice pattern
RUN apt-get update && apt-get install -y \
    curl \
    git \
    python3-pip \
    --no-install-recommends \    # only install what's needed, skip recommended extras
  && rm -rf /var/lib/apt/lists/* # delete package cache to reduce image size

# Why chain with && on one RUN:
# Each RUN is a Docker layer — separate update + install = stale cache bug
# apt-get update in layer 1, apt-get install in layer 2 = install uses old index
```

### The Stale Cache Bug (Critical Docker Interview Point)

```dockerfile
# ❌ WRONG — classic Docker caching bug
RUN apt-get update
RUN apt-get install -y nginx     # Docker may use cached layer from days ago
                                 # apt-get update result is stale

# ✅ CORRECT — always chain update + install
RUN apt-get update && apt-get install -y nginx
```

---

## YUM / DNF — RedHat / CentOS / Fedora

### DNF vs YUM

- `yum` = older tool, Python 2, CentOS 7 and earlier
- `dnf` = modern replacement, Python 3, Fedora / CentOS 8+ / RHEL 8+
- Commands are nearly identical — `dnf` is a drop-in replacement for `yum`

### Essential Commands

```bash
# ─── Update & Upgrade ─────────────────────────────────────────
dnf check-update                    # check for available updates
dnf update                          # update all packages
dnf update nginx                    # update specific package

# ─── Install & Remove ─────────────────────────────────────────
dnf install nginx                   # install a package
dnf install nginx-1.18.0            # install specific version
dnf remove nginx                    # remove package
dnf autoremove                      # remove unused dependencies

# ─── Search & Info ────────────────────────────────────────────
dnf search nginx                    # search available packages
dnf info nginx                      # show package details
dnf list installed                  # list all installed packages
dnf list installed | grep nginx     # check specific package

# ─── History & Rollback (DNF Superpower) ──────────────────────
dnf history                         # show all past transactions
dnf history info 5                  # details of transaction #5
dnf history undo 5                  # rollback transaction #5

# ─── Groups ───────────────────────────────────────────────────
dnf grouplist                       # list package groups
dnf groupinstall "Development Tools" # install a group of related packages

# ─── Cache ────────────────────────────────────────────────────
dnf clean all                       # clean all cached data
dnf makecache                       # pre-download package metadata
```

> `dnf history undo` is a powerful feature with no APT equivalent — you can roll back an entire install/update transaction. Extremely useful in production when an update breaks something.

---

## Repositories

A **repository** (repo) is a remote server hosting packages. The package manager downloads from repos defined in config files.

### APT Repository Config

```bash
# Main config file
cat /etc/apt/sources.list

# Additional repos (preferred location for custom repos)
ls /etc/apt/sources.list.d/

# Format of a sources.list entry:
# deb [options] url distribution component
deb http://archive.ubuntu.com/ubuntu focal main restricted universe multiverse
#   ^protocol  ^repo URL         ^codename ^components (sections)
```

### Adding a Custom APT Repository (Real Pattern)

```bash
# Modern way — using signed repos (Docker, Kubernetes, etc. use this)
curl -fsSL https://download.docker.com/linux/ubuntu/gpg \
  | gpg --dearmor -o /usr/share/keyrings/docker-archive-keyring.gpg

echo "deb [arch=amd64 signed-by=/usr/share/keyrings/docker-archive-keyring.gpg] \
  https://download.docker.com/linux/ubuntu $(lsb_release -cs) stable" \
  | tee /etc/apt/sources.list.d/docker.list

apt-get update && apt-get install -y docker-ce
```

### YUM/DNF Repository Config

```bash
# Repos live here — one .repo file per repository
ls /etc/yum.repos.d/

# Format of a .repo file
cat /etc/yum.repos.d/nginx.repo
```

```ini
[nginx-stable]
name=nginx stable repo
baseurl=http://nginx.org/packages/centos/$releasever/$basearch/
gpgcheck=1
enabled=1
gpgkey=https://nginx.org/keys/nginx_signing.key
```

```bash
# Add a repo directly via command
dnf config-manager --add-repo https://download.docker.com/linux/centos/docker-ce.repo

# Enable / disable a repo
dnf config-manager --enable nginx-stable
dnf config-manager --disable nginx-stable

# Install from a specific repo only
dnf install nginx --enablerepo=nginx-stable
```

### GPG Signing — Why It Matters

Every production repo should have `gpgcheck=1`. This verifies package signatures using the repo's public GPG key — ensuring packages haven't been tampered with in transit (supply chain security).

```bash
# APT — import a GPG key
apt-key add key.gpg                         # legacy method
gpg --dearmor -o /usr/share/keyrings/...   # modern method (preferred)

# YUM/DNF — import a GPG key
rpm --import https://repo.example.com/key.gpg
```

---

## Package Pinning & Version Locking (Production Critical)

In production and Docker images, you never want `apt install nginx` pulling whatever the latest version is. You pin to a specific version for **reproducibility**.

```bash
# APT — install specific version
apt-get install -y nginx=1.18.0-0ubuntu1

# APT — hold a package at current version (prevent upgrades)
apt-mark hold nginx
apt-mark unhold nginx

# DNF — install specific version
dnf install nginx-1.18.0

# DNF — lock version with versionlock plugin
dnf install dnf-plugin-versionlock
dnf versionlock add nginx
dnf versionlock list
```

> In MLOps and CI/CD, always pin package versions in Dockerfiles. Unpinned installs are a common source of "it worked yesterday, broken today" failures.

---

## Side-by-Side Cheat Sheet

| Action | APT (Ubuntu/Debian) | DNF (RHEL/Fedora) |
|---|---|---|
| Update index | `apt update` | `dnf check-update` |
| Upgrade all | `apt upgrade` | `dnf update` |
| Install | `apt install pkg` | `dnf install pkg` |
| Remove | `apt remove pkg` | `dnf remove pkg` |
| Remove + config | `apt purge pkg` | `dnf remove pkg` (configs removed too) |
| Remove orphans | `apt autoremove` | `dnf autoremove` |
| Search | `apt search pkg` | `dnf search pkg` |
| Show info | `apt show pkg` | `dnf info pkg` |
| List installed | `apt list --installed` | `dnf list installed` |
| Clean cache | `apt clean` | `dnf clean all` |
| Add repo | edit `sources.list.d/` | `dnf config-manager --add-repo` |
| Rollback | ❌ not native | `dnf history undo N` ✅ |

---

## MLOps / DevOps Real-World Patterns

### Pattern 1 — Minimal Docker Layer

```dockerfile
# Always: update + install + clean in one RUN
RUN apt-get update \
  && apt-get install -y --no-install-recommends \
     curl \
     ca-certificates \
  && rm -rf /var/lib/apt/lists/*
```

### Pattern 2 — Non-Interactive Install (No Prompts)

```bash
# Prevent interactive prompts during install (critical in scripts/CI)
DEBIAN_FRONTEND=noninteractive apt-get install -y tzdata

# Or set globally in Dockerfile
ENV DEBIAN_FRONTEND=noninteractive
```

### Pattern 3 — Check Before Install (Idempotent Scripts)

```bash
# Only install if not already present
if ! command -v nginx &> /dev/null; then
    apt-get update && apt-get install -y nginx
fi
```

### Pattern 4 — Install from Local `.deb` / `.rpm` File

```bash
# APT — install a local .deb (still resolves dependencies)
apt install ./package.deb

# dpkg — install .deb directly (no dependency resolution)
dpkg -i package.deb
apt-get install -f          # fix broken dependencies after dpkg

# RPM — install local .rpm
rpm -ivh package.rpm

# DNF — install local .rpm (resolves dependencies)
dnf install ./package.rpm
```

---

## Common Interview Questions & Answers

**Q: What's the difference between `apt update` and `apt upgrade`?**
> `apt update` refreshes the local package index (metadata about what's available). It doesn't install anything. `apt upgrade` downloads and installs newer versions of already-installed packages using that refreshed index. You must run `update` before `upgrade` or you're working with stale metadata.

**Q: What's the difference between `apt remove` and `apt purge`?**
> `remove` uninstalls the package binaries but leaves behind configuration files. `purge` removes everything including config files — use it when you want a completely clean slate, e.g. before reinstalling a misconfigured service.

**Q: Why do Dockerfiles chain `apt-get update && apt-get install` together?**
> Docker caches each `RUN` layer. If `update` and `install` are separate layers, Docker may use a cached `update` result from days ago, causing the `install` to pull stale or unavailable package versions. Chaining them in one `RUN` forces both to execute together and stay in sync.

**Q: What is `--no-install-recommends` in apt?**
> By default, APT installs both required dependencies and "recommended" packages. `--no-install-recommends` skips the recommended packages, keeping the install minimal. In Docker images this can reduce image size significantly.

**Q: DNF vs YUM — what changed?**
> DNF (Dandified YUM) replaced YUM as the default package manager in Fedora 22, RHEL 8, and CentOS 8+. Key improvements: better dependency resolution, Python 3 based, true API, faster performance, and `dnf history undo` for rollbacks. Commands are nearly identical so existing YUM knowledge transfers directly.

---

## One-Line Revision Summary

> APT manages `.deb` packages on Debian/Ubuntu via `/etc/apt/sources.list` — always chain `update + install + clean` in one Docker `RUN` layer; YUM/DNF manages `.rpm` packages on RHEL/Fedora via `/etc/yum.repos.d/` — DNF adds rollback with `history undo`; both verify package integrity via GPG signatures; always pin versions in production for reproducibility.

---

## Interview Delivery Tips

1. **Always mention the Docker caching bug** when `apt` comes up — "separate `update` and `install` layers = stale cache" is a real production footgun that impresses interviewers.
2. **`apt-get` in scripts, `apt` in terminals** — shows you know the stability difference.
3. **`--no-install-recommends` + `rm -rf /var/lib/apt/lists/*`** — mention both together as the Docker image size reduction pattern.
4. **`dnf history undo`** — this is the DNF superpower with no APT equivalent; mentioning it unprompted signals real RedHat experience.
5. **GPG verification** — frame it as supply chain security ("we verify packages haven't been tampered with in transit") rather than just a config option.
6. **Version pinning** — connect it to reproducibility: "unpinned installs are one of the most common causes of flaky CI/CD pipelines."